# 02 - Preprocessing (Member 1)
Steps: text cleaning -> missing-value handling -> duplicate removal -> label preparation (0 = real, 1 = misinformation)
-> stratified train/val/test split -> save `data/processed/<dataset>_processed.csv`.

In [ ]:
import sys, os
from pathlib import Path
# --- Colab: uncomment to mount Drive (datasets stay OUT of GitHub) ---
# from google.colab import drive; drive.mount('/content/drive')
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

In [ ]:
import pandas as pd
from src.data.preprocessing import preprocess_dataset, save_processed, clean_text
from src.data.loader import inspect_dataframe
pd.set_option("display.max_columns", 50)
NAMES = ["fakehealth", "coaid", "pubhealth"]

## Cleaning example

In [ ]:
print(clean_text("<p>Vaccines   cause X!  See https://example.com &amp; more</p>"))

## Run preprocessing (before/after counts are printed for the report)

In [ ]:
processed = {}
for name in NAMES:
    f = Path(f"data/processed/{name}_raw_standardized.csv")
    if not f.exists():
        print(f"[skip] {name}: run notebook 01 first"); continue
    print(f"\n===== {name} =====")
    raw = pd.read_csv(f)
    processed[name] = preprocess_dataset(raw, min_words=5, seed=42)
    print(save_processed(processed[name], name))

## Sanity checks

In [ ]:
for name, df in processed.items():
    s = inspect_dataframe(df, name)
    print(name, "| dup ids:", s["duplicate_ids"], "| dup text:", s["duplicate_text"], "| empty text:", s["empty_text"])
    print(df["label"].value_counts(normalize=True).round(3).to_dict(), df["split"].value_counts().to_dict())
    display(df.head(3))

## Output contract for Members 2-4
`from src.data.preprocessing import load_processed; df = load_processed("fakehealth")`

Columns: `id, dataset, subset, title, text, label, label_raw, source, n_tweets, n_replies, n_retweets,
has_source, has_engagement, has_network, split, text_len, word_count`.
`label`: 0 real / 1 fake. Text keeps original casing. Missing engagement counts are NaN (not 0);
use `has_*` flags to build the context mask.